In [1]:
import getpass, os, requests

from langchain_groq import ChatGroq
from langchain_core.documents import Document

from pathlib import Path
from dotenv import load_dotenv

In [2]:
# if "GROQ_API_KEY" not in os.environ:
#     os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API Key: ")

In [3]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [4]:
llm = ChatGroq(
    model="openai/gpt-oss-120b", 
    temperature=0.0, 
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    )

In [5]:
resp = llm.invoke("What is the capital of France?")
print(resp.content)

The capital of France is **Paris**.


### LLM Calling with System Prompt

In LangChain, we can represnt the input and output from various entitis as messages, which act as context for the models.

Message types
 - **System message** - Tells the model how to behave and provide context for interactions
 - **Human message** - Represents user input and interactions with the model
 - **AI message** - Responses generated by the model, including text content, tool calls, and metadata
 - **Tool message** - Represents the outputs of tool calls

In [6]:
response = llm.invoke(
    [
        ("system", "You are a helpful assistant that provides information about countries and other live information. "
        "Use tools when needed."),
        ("user", "What is the capital of England, and what is the weather like in the capital of England?"),
    ],
    tools=[{"type": "browser_search"}],
)

print(response.content)
print(response.tool_calls) # this might return null because because Groq's browser_search is a built-in/server-side tool. Groq executes the tool on its servers and returns the final answer to your application; you don't necessarily get a LangChain ToolMessage containing the search result in this flow. Groq explicitly describes built-in tools as server-side execution where Groq handles the tool-calling loop internally.

**Capital of England:** London  

**Current weather in London (the capital):**  
- Temperature: about **18 °C**  
- Conditions: **light rain** with a gentle breeze (winds around 19 mph from the southwest)【1†L4-L8】  
- Forecast description: “Changeable and occasionally windy, with spells of rain and showers… Maximum temperature 23 °C.”【1†L1490-L1498】

So, London is the capital of England, and right now it is cool and rainy with light rain and breezy conditions.
[]


## Building RAG Pipeline in LangChain

In [7]:
import requests
from langchain_core.documents import Document

In [8]:
WEB_URL = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt"

In [9]:
response = requests.get(WEB_URL)
response.raise_for_status() # check for response codes and raise in case of 400, 401, 404, 500, 502 etc errors

docs = [Document(page_content=response.text, metadata={'source': WEB_URL})]
print(docs)

[Document(metadata={'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt'}, page_content='# AtliqAI HR Policies\n\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.\n\n---\n\n## Employment & Onboarding\n\n### Offer and Joining Formalities\n\nUpon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a pre-joining checklist that includes submission of educational certificates, identity proof, address proof, previous employment documents, and a recent photograph. Failure to submit required documents within 7 working days of joining may result in withholding of the

In [10]:
# use this only if the model is from hugging face otherwise use the tiktoken encoder

# from transformers import AutoTokenizer
# from langchain_text_splitters import CharacterTextSplitter

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
#     tokenizer,
#     chunk_size=100,
#     chunk_overlap=0
# )

# documents = text_splitter.split_documents(docs)

In [11]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=100, chunk_overlap=0
)

documents = text_splitter.split_documents(docs)
len(documents)

/home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


44

In [13]:
documents[0]

Document(metadata={'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt'}, page_content='# AtliqAI HR Policies\n\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.\n\n---\n\n## Employment & Onboarding\n\n### Offer and Joining Formalities')